> **Student Edition (W02B)**  
> - Ejecuta los **DEMO** como guía.  
> - En **TU TURNO (1–4)** encontrarás `TODO`: debes escribir la consulta.  
> - Regla de oro: antes de un JOIN, verifica **grain** y **cardinalidad** (si no, duplicas filas).

# W03 – SQL esencial II (JOINs + CTEs) y cardinalidad práctica

## Conexión con DDIA
- **DDIA Cap. 2**: modelado + consultas; por qué las relaciones importan.
- Conexión práctica con **Cap. 3**: algunos joins se vuelven caros o peligrosos si la cardinalidad no está controlada.

## Prerrequisitos
- W02A (o dominar SELECT/WHERE/GROUP BY).
- Tener `raw_ps` disponible (el notebook puede descargar si falta).

## Objetivos
- Construir dimensiones simples (`dim_host`, `dim_discovery`).
- Usar `JOIN` (INNER/LEFT) y **demostrar** problemas de cardinalidad.
- Usar `CTE` (`WITH ...`) para estructurar consultas.
- Validar joins con conteos: evitar duplicación accidental.

## Checklist de evidencias
- [ ] Creaste `dim_host` y `dim_discovery`
- [ ] Mostraste un JOIN “malo” (duplica filas) + corrección
- [ ] 4 consultas del TU TURNO completas


In [1]:
# Setup común (cross-platform)
import sys, subprocess
from pathlib import Path
import duckdb

DB_PATH = Path("data/exoplanets.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH))

def run_module(mod: str, *args: str):
    cmd = [sys.executable, "-m", mod, *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

raw_csv = Path("data/raw/pscomppars.csv")
if not raw_csv.exists():
    run_module("./src.ingest.download_exoplanets", "--format", "csv", "--limit", "50000")

# DuckDB no permite parámetros preparados en DDL (ej. CREATE VIEW).
# Insertamos la ruta como literal SQL, escapando comillas simples.
def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

raw_csv_abs = raw_csv.resolve()
con.execute(
    f"CREATE OR REPLACE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_quote(raw_csv_abs.as_posix())})"
)
con.sql("SELECT count(*) AS n_rows FROM raw_ps").show()


┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6107 │
└────────┘



In [2]:
# (Opcional) Ver columnas disponibles en raw_ps (útil para depurar)
con.sql("DESCRIBE raw_ps").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

## DEMO (docente)


In [3]:
# DEMO 1: dimensión de hosts (1 fila por hostname) SOLO con SQL básico
# Clave: hostname
#
# Importante:
# - DISTINCT sobre (hostname, ra, ...) NO garantiza 1 fila por hostname.
# - Para forzar 1 fila por hostname SIN window functions, agregamos por hostname.
# - MAX(...) es una forma simple de "escoger un valor" y además ignora NULLs.
#
# Nota pedagógica: ejemplo didáctico. Más adelante veremos políticas de resolución más robustas.

con.execute("""
CREATE OR REPLACE TABLE dim_host_full AS
SELECT
  hostname,
  MAX(ra) AS ra
FROM raw_ps
WHERE hostname IS NOT NULL
GROUP BY hostname
""")

# Validación rápida: en una dimensión correcta, n_rows == n_keys
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT hostname) AS n_keys FROM dim_host_full").show()

┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4554 │   4554 │
└────────┴────────┘



In [4]:
con.execute("""
CREATE OR REPLACE TABLE dim_discovery AS
SELECT DISTINCT discoverymethod, disc_year
FROM raw_ps
""")
con.execute("SELECT count(*) FROM dim_discovery").fetchall()


[(141,)]

In [5]:
# DEMO 2: fact (grano = 1 fila por planeta)
con.execute("""
CREATE OR REPLACE TABLE fact_planet AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM raw_ps
WHERE pl_name IS NOT NULL
""")
con.sql("SELECT count(*) FROM fact_planet").show()


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6107 │
└──────────────┘



### DEMO 3: JOIN correcto (many-to-one)
`fact_planet` → `dim_host` debería ser muchos-a-uno. Si `dim_host` tiene hostname único, **no debe multiplicar filas**.


In [6]:
n_fact = con.execute("SELECT count(*) FROM fact_planet").fetchone()[0]
n_join = con.execute("""
SELECT count(*)
FROM fact_planet f
JOIN dim_host_full h
  ON f.hostname = h.hostname
""").fetchone()[0]
n_fact, n_join


(6107, 6107)

### DEMO 4: JOIN “malo” (duplica filas)
Error común: unirse a una tabla que **no es dimensión** (llave no única). Fabricamos una “dimensión mala” a propósito.


In [7]:
# DEMO 3: una "dimensión" MAL construida (violando 1 fila por hostname)
# Aquí NO deduplicamos: tendrá múltiples filas por hostname.
con.execute("""
CREATE OR REPLACE TABLE dim_host_full AS
SELECT hostname, ra
FROM raw_ps
WHERE hostname IS NOT NULL
""")

# Evidencia sin HAVING (solo CTE + WHERE)
con.sql("""
WITH c AS (
  SELECT hostname, COUNT(*) AS cnt
  FROM dim_host_full
  GROUP BY hostname
)
SELECT * FROM c
WHERE cnt > 1
ORDER BY cnt DESC
LIMIT 10
""").show()

┌────────────┬───────┐
│  hostname  │  cnt  │
│  varchar   │ int64 │
├────────────┼───────┤
│ KOI-351    │     8 │
│ TRAPPIST-1 │     7 │
│ HIP 41378  │     6 │
│ HD 219134  │     6 │
│ Kepler-80  │     6 │
│ HD 34445   │     6 │
│ Kepler-11  │     6 │
│ HD 110067  │     6 │
│ HD 191939  │     6 │
│ TOI-178    │     6 │
└────────────┴───────┘
  10 rows  2 columns



In [8]:
n_join_bad = con.execute("""
SELECT count(*)
FROM fact_planet f
JOIN dim_host_full h
  ON f.hostname = h.hostname
""").fetchone()[0]

n_fact, n_join_bad


(6107, 10779)

### DEMO 5: arreglar JOIN malo con CTE (deduplicación)


In [9]:
n_join_fixed = con.execute("""
WITH dim_host_fixed AS (
  SELECT DISTINCT hostname
  FROM dim_host_full
)
SELECT count(*)
FROM fact_planet f
JOIN dim_host_fixed h
  ON f.hostname = h.hostname
""").fetchone()[0]
n_fact, n_join_fixed


(6107, 6107)

## TU TURNO (práctica guiada)


### 1) LEFT JOIN y no-match: ¿cuántas filas quedan sin match en dim_host?

In [10]:
# TODO (1) LEFT JOIN y no-match
# Objetivo: ¿cuántas filas de fact_planet quedan SIN match en dim_host?
# Pistas:
# - Usa LEFT JOIN fact_planet f con dim_host h ON f.hostname = h.hostname
# - Cuenta las filas donde h.hostname IS NULL
query = """
SELECT COUNT(*) AS no_match_rows
FROM fact_planet f
LEFT JOIN dim_host_full h
  ON f.hostname = h.hostname
WHERE h.hostname IS NULL
"""
con.sql(query).show()

┌───────────────┐
│ no_match_rows │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘



### 2) CTE + ranking: por año, método #1 (más planetas)

In [11]:
# TODO (2) CTE + ranking
# Objetivo: por cada disc_year, obtener el discoverymethod #1 (más planetas) y su conteo
# Pistas:
# - Agrupa por disc_year, discoverymethod y cuenta
# - Usa una ventana: ROW_NUMBER() OVER(PARTITION BY disc_year ORDER BY n DESC)
# - Filtra rn = 1
query = """
WITH counts AS (
  SELECT
    disc_year,
    discoverymethod,
    COUNT(*) AS n
  FROM fact_planet
  WHERE disc_year IS NOT NULL
  GROUP BY disc_year, discoverymethod
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER(
      PARTITION BY disc_year
      ORDER BY n DESC
    ) AS rn
  FROM counts
)
SELECT disc_year, discoverymethod, n
FROM ranked
WHERE rn = 1
ORDER BY disc_year
"""
con.execute(query).fetchall()

[(1992, 'Pulsar Timing', 2),
 (1994, 'Pulsar Timing', 1),
 (1995, 'Radial Velocity', 1),
 (1996, 'Radial Velocity', 6),
 (1997, 'Radial Velocity', 1),
 (1998, 'Radial Velocity', 6),
 (1999, 'Radial Velocity', 13),
 (2000, 'Radial Velocity', 16),
 (2001, 'Radial Velocity', 12),
 (2002, 'Radial Velocity', 28),
 (2003, 'Radial Velocity', 21),
 (2004, 'Radial Velocity', 18),
 (2005, 'Radial Velocity', 33),
 (2006, 'Radial Velocity', 21),
 (2007, 'Radial Velocity', 34),
 (2008, 'Radial Velocity', 36),
 (2009, 'Radial Velocity', 70),
 (2010, 'Transit', 47),
 (2011, 'Transit', 79),
 (2012, 'Transit', 93),
 (2013, 'Transit', 80),
 (2014, 'Transit', 798),
 (2015, 'Transit', 99),
 (2016, 'Transit', 1432),
 (2017, 'Transit', 87),
 (2018, 'Transit', 242),
 (2019, 'Transit', 107),
 (2020, 'Transit', 165),
 (2021, 'Transit', 457),
 (2022, 'Transit', 191),
 (2023, 'Transit', 224),
 (2024, 'Transit', 187),
 (2025, 'Transit', 141),
 (2026, 'Transit', 10)]

### 3) Validación de cardinalidad: ¿hay duplicados en (discoverymethod, disc_year) en dim_discovery?

In [12]:
# TODO (3) Validación de cardinalidad
# Objetivo: detectar si hay duplicados en la clave (discoverymethod, disc_year) dentro de dim_discovery
# Pistas:
# - GROUP BY discoverymethod, disc_year
# - HAVING COUNT(*) > 1
query = """
SELECT
  discoverymethod,
  disc_year,
  COUNT(*) AS cnt
FROM dim_discovery
GROUP BY discoverymethod, disc_year
HAVING COUNT(*) > 1
"""
con.execute(query).fetchall()

[]

### 4) JOIN + agregación: promedio de RA del host por método

In [13]:
# TODO (4) JOIN + agregación
# Objetivo: para cada discoverymethod, contar planetas y calcular el promedio de RA del host.
# Nota: dim_host_full solo tiene (hostname, ra). Por eso usamos ra (no st_teff).
# Pistas:
# - JOIN fact_planet (f) con dim_host_full (h) por hostname
# - Filtra discoverymethod y ra no nulos
# - GROUP BY discoverymethod
query = """
SELECT
  f.discoverymethod,
  COUNT(*) AS n_planets,
  AVG(h.ra) AS avg_ra_host
FROM fact_planet f
JOIN dim_host_full h
  ON f.hostname = h.hostname
WHERE f.discoverymethod IS NOT NULL
  AND h.ra IS NOT NULL
GROUP BY f.discoverymethod
ORDER BY n_planets DESC
"""
con.sql(query).show()

┌───────────────────────────────┬───────────┬────────────────────┐
│        discoverymethod        │ n_planets │    avg_ra_host     │
│            varchar            │   int64   │       double       │
├───────────────────────────────┼───────────┼────────────────────┤
│ Transit                       │      7963 │  250.8860938951409 │
│ Radial Velocity               │      2240 │ 175.21128098415085 │
│ Microlensing                  │       284 │  267.3787386570424 │
│ Imaging                       │       115 │ 199.36121578956522 │
│ Transit Timing Variations     │       106 │  258.2377605773584 │
│ Eclipse Timing Variations     │        31 │ 232.07414613870966 │
│ Orbital Brightness Modulation │        17 │ 291.36825150588237 │
│ Pulsar Timing                 │        14 │ 208.57388799999995 │
│ Astrometry                    │         6 │ 235.42183843333336 │
│ Pulsation Timing Variations   │         2 │       315.24251065 │
│ Disk Kinematics               │         1 │        167.01334

## DEMO (capstone, opcional) — CTE + Window Functions (estilo “heroes”)

> **Esta sección es opcional y NO entra en el entregable obligatorio.**  
> La idea es cerrar la teoría con un ejemplo “potente” como el que vimos con `heroes/team`:
> - 1er CTE: limpiamos/filtramos filas (como `heroes_clean`)
> - 2do CTE: calculamos métricas por grupo con **window functions** y rankeamos (como `ranked`)
> - SELECT final: nos quedamos con **la fila `rn=1`** por grupo (el “top 1” por partición)

**Pregunta análoga (en exoplanetas):**  
Para cada `hostname` (sistema), encontrar el **planeta con mayor radio** (`pl_rade`) y además reportar:
- `avg_radius_in_system` (promedio de radios en el sistema)
- `planets_in_system` (cuántos planetas tiene el sistema)
- un atributo del host (`ra`, si existe)

In [14]:
# CTE + Window Functions (capstone)
query = r'''
WITH planets_clean AS (
  SELECT
    f.pl_name,
    f.hostname,
    f.pl_rade,
    h.ra
  FROM fact_planet f
  LEFT JOIN dim_host_full h
    ON h.hostname = f.hostname
  WHERE f.hostname IS NOT NULL
    AND f.pl_rade IS NOT NULL
    AND f.pl_rade BETWEEN 0 AND 30   -- filtro tipo "edad entre 0 y 120"
),
ranked AS (
  SELECT
    hostname,
    ra,
    pl_name,
    pl_rade,
    AVG(pl_rade) OVER (PARTITION BY hostname) AS avg_radius_in_system,
    COUNT(*)     OVER (PARTITION BY hostname) AS planets_in_system,
    ROW_NUMBER() OVER (
      PARTITION BY hostname
      ORDER BY pl_rade DESC, pl_name ASC
    ) AS rn
  FROM planets_clean
)
SELECT
  hostname,
  ra,
  pl_name AS largest_planet,
  pl_rade AS largest_radius,
  avg_radius_in_system,
  planets_in_system
FROM ranked
WHERE rn = 1
ORDER BY planets_in_system DESC, hostname
LIMIT 50;
'''
con.sql(query).show()

┌────────────────┬─────────────┬────────────────┬────────────────┬──────────────────────┬───────────────────┐
│    hostname    │     ra      │ largest_planet │ largest_radius │ avg_radius_in_system │ planets_in_system │
│    varchar     │   double    │    varchar     │     double     │        double        │       int64       │
├────────────────┼─────────────┼────────────────┼────────────────┼──────────────────────┼───────────────────┤
│ KOI-351        │ 284.4334642 │ KOI-351 h      │         11.252 │    3.899999999999998 │                64 │
│ TRAPPIST-1     │ 346.6263919 │ TRAPPIST-1 g   │          1.129 │   0.9785714285714291 │                49 │
│ HD 110067      │ 189.8392243 │ HD 110067 d    │          2.852 │    2.431333333333333 │                36 │
│ HD 191939      │ 302.0256247 │ HD 191939 f    │           13.2 │                 6.59 │                36 │
│ HD 219134      │ 348.3372026 │ HD 219134 h    │           12.7 │   3.6688333333333336 │                36 │
│ HD 34445

## Para entregar (W03) 

### En clase
1) `docs/w03_sql_practice.md` con el análisis de `dim_host_full`:
   - Los 4 conteos: `n_fact`, `n_join_good`, `n_join_bad`, `n_join_fixed` + 2–3 líneas explicando qué pasó.
   - Respuestas a **TODO 1–4** (cada una: SQL + output pegado).

2) `docs/decisions_log.md`: 1 entrada corta:
   - “Cómo validé cardinalidad antes de un JOIN” + evidencia (una query de duplicados o un conteo).

> **Nota:** La sección **DEMO capstone (CTE + Window)** es **solo demostración** (no se entrega).

### Tarea (para la próxima clase)
1) `docs/w03_join_case.md`: **1 caso real de JOIN malo**
   - evidencia con conteos antes/después
   - diagnóstico (qué clave falló)
   - fix simple (por ejemplo: dedupe con `GROUP BY`/`DISTINCT` o pre-agregación)

2) 2 consultas extra (en `docs/w03_sql_practice.md`):
   - 1 que incluya `JOIN`
   - 1 que incluya `CTE`

## SOLUCIÓN TAREA

En esta versión armé un caso de JOIN problemático usando una dimensión derivada con cardinalidad no controlada. La idea es mostrar cómo una clave aparentemente razonable puede inflar filas si no se valida antes.


### Parte 1: caso de JOIN malo

Primero construyo una tabla auxiliar a partir del dataset base. A propósito no la deduplico completamente para poder observar el efecto del JOIN.


In [15]:
con.execute('''
CREATE OR REPLACE TABLE dim_host_year_bad AS
SELECT
  hostname,
  disc_year
FROM fact_planet
WHERE hostname IS NOT NULL
  AND disc_year IS NOT NULL
''')

con.execute('''
CREATE OR REPLACE TABLE dim_host_year_fixed AS
SELECT
  hostname,
  MIN(disc_year) AS first_disc_year
FROM dim_host_year_bad
GROUP BY hostname
''')


In [16]:
n_fact = con.execute('SELECT COUNT(*) FROM fact_planet').fetchone()[0]
n_join_bad = con.execute('''
SELECT COUNT(*)
FROM fact_planet f
JOIN dim_host_year_bad d
  ON f.hostname = d.hostname
''').fetchone()[0]
n_join_fixed = con.execute('''
SELECT COUNT(*)
FROM fact_planet f
JOIN dim_host_year_fixed d
  ON f.hostname = d.hostname
''').fetchone()[0]

print('n_fact:', n_fact)
print('n_join_bad:', n_join_bad)
print('n_join_fixed:', n_join_fixed)
print('inflation_bad:', n_join_bad - n_fact)


n_fact: 6107
n_join_bad: 10775
n_join_fixed: 6107
inflation_bad: 4668


In [17]:
# Evidencia de la causa: hosts repetidos en la dimensión mala
con.execute('''
SELECT
  hostname,
  COUNT(*) AS repeats
FROM dim_host_year_bad
GROUP BY hostname
HAVING COUNT(*) > 1
ORDER BY repeats DESC, hostname ASC
LIMIT 10
''').fetchdf()


,hostname,repeats
0,KOI-351,8
1,TRAPPIST-1,7
2,HD 10180,6
3,HD 110067,6
4,HD 191939,6
5,HD 219134,6
6,HD 34445,6
7,HIP 41378,6
8,K2-138,6
9,Kepler-11,6


**Diagnóstico:** el JOIN malo ocurre porque `dim_host_year_bad` tiene múltiples filas por `hostname`. Entonces, al unir solo por esa clave, algunas filas de `fact_planet` se multiplican. El fix consiste en preagregar o deduplicar para asegurar una fila por host antes del JOIN.


### Parte 2: consulta extra con JOIN

Ahora uso la versión corregida para enriquecer la tabla de hechos con el primer año de descubrimiento observado por sistema.


In [18]:
query = '''
SELECT
  f.discoverymethod,
  COUNT(*) AS n_planets,
  MIN(d.first_disc_year) AS oldest_host_year,
  MAX(d.first_disc_year) AS newest_host_year
FROM fact_planet f
LEFT JOIN dim_host_year_fixed d
  ON f.hostname = d.hostname
WHERE f.discoverymethod IS NOT NULL
GROUP BY f.discoverymethod
ORDER BY n_planets DESC
LIMIT 10
'''
con.execute(query).fetchdf()


,discoverymethod,n_planets,oldest_host_year,newest_host_year
0,Transit,4501,2001,2026
1,Radial Velocity,1166,1995,2026
2,Microlensing,266,2004,2026
3,Imaging,92,2004,2026
4,Transit Timing Variations,39,2009,2025
5,Eclipse Timing Variations,17,2009,2023
6,Orbital Brightness Modulation,9,2011,2021
7,Pulsar Timing,8,1992,2024
8,Astrometry,6,2013,2025
9,Pulsation Timing Variations,2,2007,2016


### Parte 3: consulta extra con CTE

Aquí uso un CTE para resumir sistemas y luego filtrar solo aquellos con más de un planeta reportado y con radio promedio disponible.


In [19]:
query = '''
WITH host_summary AS (
  SELECT
    hostname,
    COUNT(*) AS n_planets,
    AVG(pl_rade) AS avg_radius,
    AVG(pl_orbper) AS avg_orbper
  FROM fact_planet
  WHERE hostname IS NOT NULL
  GROUP BY hostname
)
SELECT
  hostname,
  n_planets,
  avg_radius,
  avg_orbper
FROM host_summary
WHERE n_planets >= 2
  AND avg_radius IS NOT NULL
ORDER BY n_planets DESC, avg_radius DESC
LIMIT 10
'''
con.execute(query).fetchdf()


,hostname,n_planets,avg_radius,avg_orbper
0,KOI-351,8,3.900000,106.138118
1,TRAPPIST-1,7,0.978571,7.773692
2,HD 34445,6,8.855000,1301.252500
3,HD 10180,6,6.676667,500.713115
4,HD 191939,6,6.590000,559.822184
5,HIP 41378,6,4.253667,216.463277
6,HD 219134,6,3.668833,403.438918
7,TOI-1136,6,3.051819,17.936167
8,Kepler-11,6,2.966667,40.513600
9,K2-138,6,2.584333,12.384155


### Decisión para `docs/decisions_log.md`

```markdown
- Fecha: 2026-04-04
- Decisión: Validar unicidad de la clave de unión antes de conectar una tabla de hechos con una dimensión derivada.
- Razón: Una dimensión con varias filas por clave puede inflar silenciosamente los resultados y alterar agregaciones posteriores.
- Alternativas: unir directamente y revisar después (rechazada), confiar en que la dimensión ya estaba limpia (rechazada).
- Evidencia: comparación entre `n_fact`, `n_join_bad` y `n_join_fixed`, más conteo de hosts repetidos en `dim_host_year_bad`.
```
